In [ ]:
# ==========================================================
# TAX TOOL - VERSION 1.2
#
# FILE EXCEL:
#   Tong hop MST.xlsx
#
# PDF:
#   - Mẫu 14-MST (Cấp MST)
#   - Mẫu 40/TB-ĐKT (Trùng MST)
#
# OUTPUT:
#   Tong_hop_MST_Updated.xlsx
# ==========================================================

!pip install pymupdf openpyxl -q

import re
import fitz
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

# ==========================================================
# UPLOAD FILE
# ==========================================================

print("UPLOAD:")
print("- 1 file Tong hop MST.xlsx")
print("- Toàn bộ file PDF từ Cục Thuế\n")

uploaded = files.upload()

excel_file = None
pdf_files = []

for file in uploaded:
    if file.lower().endswith((".xlsx", ".xls")):
        excel_file = file
    elif file.lower().endswith(".pdf"):
        pdf_files.append(file)

if excel_file is None:
    raise Exception("Không tìm thấy file Excel.")

print("\nFILE EXCEL:")
print(excel_file)

print("\nPDF:")
for pdf in pdf_files:
    print("-", pdf)

# ==========================================================
# HÀM ĐỌC PDF
# ==========================================================

def read_pdf(path):
    text = ""
    doc = fitz.open(path)
    for page in doc:
        text += page.get_text("text", sort=True)
    return text


# ==========================================================
# LẤY MST TỪ MẪU 14
# ==========================================================

def parse_14(text, pdf_name):
    results = {}

    cccd_list = re.findall(r'\b\d{12}\b', text)

    for cccd in cccd_list:
        results[cccd] = {
            "mst": cccd,
            "status": "CAP_MOI",
            "note": "Được cấp MST",
            "source": pdf_name
        }

    return results


# ==========================================================
# LẤY MST TỪ MẪU 40 (ĐÃ FIX LỖI NGẮT TRANG)
# ==========================================================

def parse_40(text, pdf_name):
    results = {}
    text = text.replace("\n", " ")

    # CẬP NHẬT Regex:
    # Cho phép quét qua các số lẻ tẻ (như số trang 3/11) giữa chữ MST và mã 10 số
    # Nhưng tuyệt đối không được nhảy qua một số CCCD (12 số) khác
    pattern = r'\b(\d{12})\b(?:(?!\b\d{12}\b).)*?MST(?:(?!\b\d{12}\b).)*?(\b\d{10}\b)'

    matches = re.findall(
        pattern,
        text,
        flags=re.DOTALL
    )

    for cccd, mst in matches:
        results[cccd] = {
            "mst": mst,
            "status": "TRUNG_MST",
            "note": f"MST đã tồn tại: {mst}",
            "source": pdf_name
        }

    return results

# ==========================================================
# ĐỌC TOÀN BỘ PDF
# ==========================================================

all_results = {}

print("\nĐANG ĐỌC PDF...\n")

for pdf in pdf_files:
    text = read_pdf(pdf)

    if "14-MST" in text or "14- MST" in text:
        data = parse_14(text, pdf)

        for cccd, value in data.items():
            if cccd not in all_results:
                all_results[cccd] = value
            elif all_results[cccd]["status"] == "CAP_MOI":
                continue
            elif value["status"] == "CAP_MOI":
                all_results[cccd] = value

        print(f"{pdf:<30} -> {len(data)} hồ sơ cấp MST")

    elif "40/TB" in text:
        data = parse_40(text, pdf)

        for cccd, value in data.items():
            if cccd not in all_results:
                all_results[cccd] = value
            elif all_results[cccd]["status"] == "CAP_MOI":
                continue
            elif value["status"] == "CAP_MOI":
                all_results[cccd] = value

        print(f"{pdf:<30} -> {len(data)} hồ sơ trùng MST")

    else:
        print(f"{pdf:<30} -> Không nhận diện")

print("\nTỔNG SỐ HỒ SƠ LẤY ĐƯỢC:", len(all_results))

# ==========================================================
# ĐỌC EXCEL
# ==========================================================

print("\nĐang mở Excel...")
wb = load_workbook(excel_file)
ws = wb.active

# ==========================================================
# XÁC ĐỊNH CỘT TRẠNG THÁI, GHI CHÚ VÀ FILE NGUỒN
# ==========================================================

last_col = ws.max_column

status_col = get_column_letter(last_col + 1)
note_col = get_column_letter(last_col + 2)
source_col = get_column_letter(last_col + 3)

print(f"TRANG_THAI -> {status_col}")
print(f"GHI_CHU    -> {note_col}")
print(f"FILE_NGUON -> {source_col}")

# Header (dòng 3 là dòng tiêu đề của file)
ws[f"{status_col}3"] = "TRANG_THAI"
ws[f"{note_col}3"] = "GHI_CHU"
ws[f"{source_col}3"] = "FILE_NGUON"

# ==========================================================
# MÀU TÔ
# ==========================================================

green_fill = PatternFill(
    fill_type="solid",
    start_color="C6EFCE"
)

yellow_fill = PatternFill(
    fill_type="solid",
    start_color="FFF2CC"
)

# ==========================================================
# CẬP NHẬT
# ==========================================================

updated = 0
duplicate = 0

for row in range(4, ws.max_row + 1):

    cccd = ws[f"M{row}"].value

    if cccd is None:
        continue

    # Loại bỏ khoảng trắng, chữ cái, hoặc ký tự ẩn từ file Excel
    cccd = str(cccd)
    cccd = re.sub(r'\D', '', cccd).zfill(12)

    if cccd in all_results:
        result = all_results[cccd]

        # Cập nhật MST
        ws[f"D{row}"] = result["mst"]

        # Trạng thái
        ws[f"{status_col}{row}"] = result["status"]

        # Ghi chú
        ws[f"{note_col}{row}"] = result["note"]

        # File nguồn
        ws[f"{source_col}{row}"] = result["source"]

        # Tô màu
        if result["status"] == "CAP_MOI":
            ws[f"D{row}"].fill = green_fill
        else:
            ws[f"D{row}"].fill = yellow_fill
            duplicate += 1

        updated += 1


# ==========================================================
# SAVE
# ==========================================================

output = "Tong_hop_MST_Updated.xlsx"
wb.save(output)

print("\n==============================")
print("HOÀN THÀNH")
print("==============================")
print(f"Tổng cập nhật : {updated}")
print(f"Trùng MST     : {duplicate}")
print(f"Không tìm thấy: {ws.max_row-3-updated}")
print("==============================")

# ==========================================================
# DOWNLOAD
# ==========================================================

files.download(output)